# Module: Vector Databases in RAG (Chroma, Qdrant, Milvus, pgvector)

Vector databases are purpose-built to store, index, and query high-dimensional vector embeddings efficiently. In a RAG pipeline, they replace traditional keyword lookups with Approximate Nearest Neighbor (ANN) search, mapping queries to the closest document chunks in milliseconds.

## 1. Chroma (The Prototype & Developer-Friendly Option)

Chroma is built to be lightweight, developer-friendly, and embedded directly into your application code.  

Architecture: Can run entirely in-memory or persisted locally to SQLite. It also features a client-server mode.

Best For: Local development, rapid prototyping, notebooks, and small-to-medium single-tenant apps.  

Pros: Incredibly easy to initialize (pip install chromadb), minimal configuration, native LangChain/LlamaIndex integration.  

Cons: Not natively designed for massive horizontal distributed scaling out of the box.Python Implementation:

In [ ]:
import chromadb

# Initialize persistent client
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="rag_documents")

# Add chunks
collection.add(
    documents=["RAG combines search and generation."],
    embeddings=[[0.1, 0.2, 0.3]], # or let Chroma handle it natively
    ids=["doc1"]
)

# Query
results = collection.query(
    query_embeddings=[[0.1, 0.2, 0.3]],
    n_results=1
)

## 2. Qdrant (The Rust-Powered, Filter-First Engine)
Qdrant is written in Rust, prioritizing raw search performance and advanced metadata filtering.  

Architecture: Standalone service deployed via Docker or Kubernetes. Employs optimized graph-based indexing (HNSW).

Best For: Production systems requiring complex payload/metadata filtering combined with vector search (e.g., "Find chunks similar to X, but only from documents modified after 2025 with security tag 'public'").

Pros: Lightning-fast search speeds, rich payload filtering engine, support for scalar quantization (reduces memory consumption drastically).  

Cons: Requires managing a standalone infrastructure service.  Python Implementation:

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance

client = QdrantClient(":memory:") # or host="localhost", port=6333
client.create_collection(
    collection_name="rag_docs",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE)
)

## 3. Milvus (The Billion-Scale Enterprise Powerhouse)
Milvus is a cloud-native, highly distributed vector database engineered to handle massive, web-scale datasets. 

Architecture: Disaggregated storage and compute. Separates data nodes, index nodes, query nodes, and object storage (e.g., S3).  

Best For: Enterprise environments scaling past tens or hundreds of millions of vectors.

Pros: Extreme horizontal scalability, robust multi-tenancy, and high availability.

Cons: Heavy operational complexity; overkill for small or mid-sized applications.  Python Implementation

In [ ]:
from pymilvus import MilvusClient

client = MilvusClient("milvus_demo.db")
client.create_collection(
    collection_name="rag_collection",
    dimension=1536
)

## 4. pgvector (The All-In-One PostgreSQL Extension)

pgvector turns the standard PostgreSQL relational database into a vector database, allowing you to run vector math side-by-side with your relational tables.  

Architecture: A native C-extension installed directly into PostgreSQL. Supports HNSW and IVFFlat indices.  

Best For: Teams already running PostgreSQL who want to avoid adding a brand-new database to their infrastructure stack.  

Pros: ACID compliance, unified backups, single operational stack, transactional consistency between business data and vectors.  

Cons: Scaling past tens of millions of rows requires careful PostgreSQL performance tuning and maintenance.
SQL/Python Implementation

In [ ]:
-- Enable extension inside Postgres
CREATE EXTENSION vector;

-- Create table with vector column
CREATE TABLE chunks (id SERIAL PRIMARY KEY, content TEXT, embedding vector(1536));

-- Create HNSW index for fast search
CREATE INDEX ON chunks USING hnsw (embedding vector_cosine_ops);

-- Query via SQL
SELECT content FROM chunks ORDER BY embedding <=> '[0.01, 0.02, ...]' LIMIT 5;

### Quick Comparison Matrix

| Feature | Chroma | Qdrant | Milvus | pgvector
| :--- | :--- | :--- | :--- | :--- |
| Primary Deployment | Embedded / Local | Standalone (Rust) | Distributed Cloud-Native | Postgres Extension
| Scale Target | Prototypes / Small (<100k) | Mid-to-Large Scale | Billion-Scale | Up to ~50M Chunks
| Operational Overhead | Extremely Low | Moderate | High | None (if using Postgres)
| Filtering Capabilities | Basic | Advanced / Filter-First | Advanced | Relational SQL Joins